# 01 - Exploring BCCD and the segmentation pipeline

Interactive companion to the scripts. Run `bash scripts/run_all.sh` first so that `results/` is populated; this notebook only reads results and re-runs the pipeline on a **training** image (never tunes on test).

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import segment as S
from src.loader import draw_boxes, load_image
from src.preprocess import preprocess
from src.utils import FIG_DIR, TABLE_DIR, load_splits

pd.set_option('display.width', 200)


def show(name, width=14):
    img = plt.imread(FIG_DIR / name)
    plt.figure(figsize=(width, width * img.shape[0] / img.shape[1]))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## 1. Dataset after cleaning

Counts per split and the cleaning log (from `src/loader.py`).

In [ ]:
print(pd.read_csv(TABLE_DIR / 'dataset_stats.csv').to_string(index=False))
print(json.load(open(ROOT / 'results/logs/cleaning_log.json')))
show('gt_box_overlays.png')

## 2. Preprocessing on one training image

In [ ]:
splits = load_splits()
ann = pd.read_csv(TABLE_DIR / 'annotations_clean.csv')
image_id = splits['train'][5]
rgb = load_image(image_id)
pre = preprocess(rgb)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, key, cmap in zip(axes, ['rgb', 'clahe', 'normalized', 'purple'], [None, 'gray', 'gray', 'magma']):
    ax.imshow(pre[key], cmap=cmap)
    ax.set_title(key)
    ax.axis('off')
plt.show()

## 3. The four segmenters with the tuned parameters (training image)

Parameters come from `results/tables/seg_params.json` (selected on train, checked on val).

In [ ]:
cfg = json.load(open(TABLE_DIR / 'seg_params.json'))
params = cfg['params']
print('best method on val:', cfg['best_method_by_val_f1@0.5'])
prep = S.prepare(rgb, **params)
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
axes[0].imshow(draw_boxes(rgb, ann[ann.image == image_id], label=False))
axes[0].set_title('GT boxes')
for ax, method in zip(axes[1:], S.METHODS):
    seg = S.segment_prepared(prep, method, **params)
    ax.imshow(seg.labels, cmap='nipy_spectral', interpolation='nearest')
    ax.set_title(f'{method}: {len(seg.types)} regions')
for ax in axes:
    ax.axis('off')
plt.show()

## 4. Segmentation results on the test split

Precision is a **lower bound**: BCCD annotators skipped many RBCs, so some false positives are real cells.

In [ ]:
print(pd.read_csv(TABLE_DIR / 'segmentation_comparison.csv').set_index('method').T.round(3).to_string())
show('segmentation_grid_test.png')

## 5. Sensitivity: fixed vs adaptive watershed under rescaling and colour shifts

In [ ]:
sens = pd.read_csv(TABLE_DIR / 'sensitivity.csv')
print(sens.pivot(index='condition', columns='method', values='f1@0.5').round(3).to_string())
print(pd.read_csv(TABLE_DIR / 'sensitivity_area_cutoff_check.csv').to_string(index=False))
show('sensitivity.png')

## 6. Classification (GT crops vs segmented regions) and leakage checks

In [ ]:
clf = pd.read_csv(TABLE_DIR / 'classification_comparison.csv')
print(clf.drop(columns=['best_params']).round(3).to_string(index=False))
print(pd.read_csv(TABLE_DIR / 'leakage_checks.csv').to_string(index=False))
show('confusion_matrices.png')

## 7. Interpretability

In [ ]:
print(pd.read_csv(TABLE_DIR / 'permutation_importance.csv').head(10).round(4).to_string(index=False))
show('permutation_importance.png', width=7)
show('class_feature_means.png')